In [ ]:
!pip install -q transformers torch bitsandbytes

In [ ]:
import time
import torch
import numpy as np
from threading import Thread
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

In [ ]:
device = "cuda"
model_name = "Qwen/Qwen3-0.6B" 

In [ ]:
print("Dang tai model...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=quantization_config,
    torch_dtype="auto",
    device_map=device
)
print("Load model thanh cong!\n")

In [ ]:
def run_benchmark(prompt, num_requests=100):
    print(f"Bat dau test tai voi {num_requests} requests...")
    print(f"Prompt: {prompt}\n")
    
    # Lưu trữ kết quả
    ttft_list = [] # Time to first token (ms)
    tps_list = []  # Tokens per second
    
    # Chuẩn bị input (Tokenize 1 lần để dùng lại nếu prompt không đổi)
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([input_text], return_tensors="pt").to(device)

    # Warmup (Chạy mồi 1 lần để load kernels vào GPU, không tính vào thống kê)
    print("Dang chay Warmup (khong tinh vao ket qua)...")
    _streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    _kwargs = dict(model_inputs, streamer=_streamer, max_new_tokens=50)
    _t = Thread(target=model.generate, kwargs=_kwargs)
    _t.start()
    _t.join() # Đợi warmup xong
    print("Warmup xong. Bat dau do luong...\n")

    # Vòng lặp test
    for i in range(num_requests):
        # Streamer giúp nhận token ngay khi model sinh ra
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        generation_kwargs = dict(
            model_inputs, 
            streamer=streamer, 
            max_new_tokens=512,
            pad_token_id=tokenizer.eos_token_id
        )

        # Chạy model.generate trong một luồng riêng để không chặn luồng chính (main thread)
        thread = Thread(target=model.generate, kwargs=generation_kwargs)
        
        # Bắt đầu đo giờ
        start_time = time.perf_counter()
        thread.start()

        first_token_received = False
        first_token_time = 0
        generated_text = ""

        # Lặp qua các token được sinh ra từ streamer
        for new_text in streamer:
            # Bắt thời điểm token đầu tiên xuất hiện
            if not first_token_received:
                first_token_time = time.perf_counter() - start_time
                first_token_received = True
            
            generated_text += new_text

        end_time = time.perf_counter()
        
        # Tính toán chỉ số cho request này
        # Tính lại số token thực tế sinh ra để tính tốc độ chính xác
        num_new_tokens = len(tokenizer.encode(generated_text))
        
        # Thời gian sinh (Decode time) = Tổng thời gian - Thời gian prefill (TTFT)
        generate_time = end_time - (start_time + first_token_time)
        
        # Tốc độ sinh (Tokens/s)
        current_tps = num_new_tokens / generate_time if generate_time > 0 else 0
        
        # Chuyển TTFT sang ms
        current_ttft_ms = first_token_time * 1000

        ttft_list.append(current_ttft_ms)
        tps_list.append(current_tps)

        print(f"Request {i+1}/{num_requests} | Tokens: {num_new_tokens} | TTFT: {current_ttft_ms:.2f}ms | Speed: {current_tps:.2f} tok/s")

    return ttft_list, tps_list


In [ ]:
    prompt_test = "Tell me about AI in today economic"
    
    # Chạy test
    ttfts, tpss = run_benchmark(prompt_test, num_requests=100)

    # Tính toán thống kê
    avg_ttft = np.mean(ttfts)
    p95_ttft = np.percentile(ttfts, 95) 
    
    avg_tps = np.mean(tpss)
    
    print("\n" + "="*40)
    print("KET QUA BENCHMARK (100 Requests)")
    print("="*40)
    print(f"Model: {model_name}")
    print(f"Thoi gian token dau tien (TTFT):")
    print(f"  - Trung binh: {avg_ttft:.2f} ms")
    print(f"  - P95 (cham nhat 5%): {p95_ttft:.2f} ms")
    print("-" * 40)
    print(f"Toc do sinh token (Throughput):")
    print(f"  - Trung binh: {avg_tps:.2f} tokens/s")
    print("="*40)